# TLoT Experiment 0: Does Φ Exist?

**The most important experiment in the TLoT program.**

We're testing whether there is a clean, separable direction in LLM activation space that corresponds to "wrong reasoning" vs "correct reasoning".

If YES → we can project it out during inference (TLoT works)  
If NO → TLoT needs fundamental rethinking

---

**What this notebook does:**
1. Generates correct/incorrect arithmetic statements
2. Extracts residual stream activations per layer
3. Finds contrastive directions (mean-diff AND logistic regression)
4. Tests stability, separability, random baseline, cross-operation generalization
5. Tests cross-domain transfer (arithmetic → factual claims)
6. Generates 6+ diagnostic plots

**Runtime:** ~5-10 min on Colab T4 with Pythia-410M

## 0. Setup

In [ ]:
# Install dependencies
!pip install -q torch transformer_lens transformers tokenizers matplotlib numpy scikit-learn

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# ─── CONFIG ───────────────────────────────────────────
# Change these to experiment with different models

MODEL_NAME = "pythia-410m"  # Options: "pythia-160m", "pythia-410m", "pythia-1.4b"
N_SAMPLES = 200            # samples per arithmetic operation
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Model: {MODEL_NAME}")
print(f"Device: {DEVICE}")
print(f"Samples per operation: {N_SAMPLES}")

## 1. Data Generation

In [ ]:
import random
import numpy as np
from dataclasses import dataclass
from typing import Optional

@dataclass
class Sample:
    prompt: str
    is_correct: bool
    operation: str
    answer: int
    wrong_answer: Optional[int] = None

def generate_arithmetic_data(n_per_type=200, seed=42):
    rng = random.Random(seed)
    samples = []
    for op, symbol in [("addition", "+"), ("multiplication", "*"), ("subtraction", "-")]:
        for _ in range(n_per_type):
            a, b = rng.randint(2, 99), rng.randint(2, 99)
            correct = eval(f"{a}{symbol}{b}")
            wrong = correct + rng.choice([-3,-2,-1,1,2,3,5,7,10])
            samples.append(Sample(f"{a} {symbol} {b} = {correct}", True, op, correct))
            samples.append(Sample(f"{a} {symbol} {b} = {wrong}", False, op, correct, wrong))
    return samples

def generate_citation_data(n=50, seed=42):
    """Real vs fabricated facts — tests cross-domain transfer."""
    rng = random.Random(seed)
    real = [
        "The capital of France is Paris.",
        "Water boils at 100 degrees Celsius at sea level.",
        "The Earth orbits the Sun.",
        "DNA has a double helix structure.",
        "Light travels at approximately 300,000 km per second.",
        "The human body has 206 bones.",
        "Shakespeare wrote Hamlet.",
        "Oxygen has atomic number 8.",
        "Pi is approximately 3.14159.",
        "Gold has the chemical symbol Au.",
    ]
    fake = [
        "The capital of France is Lyon.",
        "Water boils at 90 degrees Celsius at sea level.",
        "The Sun orbits the Earth.",
        "DNA has a triple helix structure.",
        "Light travels at approximately 500,000 km per second.",
        "The human body has 312 bones.",
        "Shakespeare wrote The Odyssey.",
        "Oxygen has atomic number 12.",
        "Pi is approximately 3.17320.",
        "Gold has the chemical symbol Gd.",
    ]
    samples = []
    for _ in range(n):
        idx = rng.randint(0, len(real)-1)
        samples.append(Sample(real[idx], True, "citation", 0))
        samples.append(Sample(fake[idx], False, "citation", 0, 1))
    return samples

samples = generate_arithmetic_data(N_SAMPLES)
print(f"Generated {len(samples)} arithmetic samples")
print(f"  Correct: {sum(1 for s in samples if s.is_correct)}")
print(f"  Wrong:   {sum(1 for s in samples if not s.is_correct)}")
print(f"\nExamples:")
for s in samples[:4]:
    print(f"  {'✓' if s.is_correct else '✗'} {s.prompt}")

## 2. Load Model & Extract Activations

In [ ]:
import transformer_lens as tl

print(f"Loading {MODEL_NAME}...")
model = tl.HookedTransformer.from_pretrained(
    MODEL_NAME, device=DEVICE,
    dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
)
print(f"Loaded: {model.cfg.n_layers} layers, d_model={model.cfg.d_model}")

N_LAYERS = model.cfg.n_layers
D_MODEL = model.cfg.d_model

In [ ]:
# Extract residual stream at last token position, all layers
all_prompts = [s.prompt for s in samples]
is_correct = [s.is_correct for s in samples]
operations = [s.operation for s in samples]

activations = {l: [] for l in range(N_LAYERS)}
BATCH_SIZE = 64

print(f"Extracting activations from {len(all_prompts)} prompts...")
for batch_start in range(0, len(all_prompts), BATCH_SIZE):
    batch = all_prompts[batch_start:batch_start+BATCH_SIZE]
    _, cache = model.run_with_cache(
        batch,
        names_filter=lambda name: "hook_resid_post" in name,
    )
    for l in range(N_LAYERS):
        act = cache[f"blocks.{l}.hook_resid_post"][:, -1, :].float().cpu()
        activations[l].append(act)
    del cache
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    if (batch_start // BATCH_SIZE) % 3 == 0:
        print(f"  {min(batch_start+BATCH_SIZE, len(all_prompts))}/{len(all_prompts)}")

for l in range(N_LAYERS):
    activations[l] = torch.cat(activations[l], dim=0)

correct_mask = torch.tensor(is_correct)
wrong_mask = ~correct_mask

op_masks = {}
for op in ["addition", "multiplication", "subtraction"]:
    op_masks[op] = torch.tensor([o == op for o in operations])

print(f"Done! Shape per layer: {activations[0].shape}")

## 3. Analyze Each Layer: Find Φ

In [ ]:
from sklearn.linear_model import LogisticRegression

def analyze_layer(layer, act_c, act_w, n_random=10, seed=42):
    """Find contrastive direction, test stability, LDA, random baseline."""
    rng = np.random.RandomState(seed)
    n = min(len(act_c), len(act_w))
    ac = act_c[:n].numpy()
    aw = act_w[:n].numpy()
    d = ac.shape[1]

    # 1. Mean-diff direction
    direction = aw.mean(0) - ac.mean(0)
    direction /= (np.linalg.norm(direction) + 1e-8)

    # 2. Split-half stability
    idx = rng.permutation(n)
    h = n // 2
    d_a = aw[idx[:h]].mean(0) - ac[idx[:h]].mean(0)
    d_a /= (np.linalg.norm(d_a) + 1e-8)
    d_b = aw[idx[h:2*h]].mean(0) - ac[idx[h:2*h]].mean(0)
    d_b /= (np.linalg.norm(d_b) + 1e-8)
    stability = float(np.dot(d_a, d_b))

    # 3. Mean-diff probe accuracy (train/test)
    train_dir = aw[idx[:h]].mean(0) - ac[idx[:h]].mean(0)
    train_dir /= (np.linalg.norm(train_dir) + 1e-8)
    tc, tw = ac[idx[h:2*h]] @ train_dir, aw[idx[h:2*h]] @ train_dir
    thresh = (tc.mean() + tw.mean()) / 2
    mean_acc = float(((tc < thresh).mean() + (tw >= thresh).mean()) / 2)

    # 4. LDA (logistic regression)
    X_train = np.vstack([ac[idx[:h]], aw[idx[:h]]])
    y_train = np.array([0]*h + [1]*h, dtype=float)
    X_test = np.vstack([ac[idx[h:2*h]], aw[idx[h:2*h]]])
    y_test = np.array([0]*(n-h) + [1]*(n-h), dtype=float)

    try:
        clf = LogisticRegression(max_iter=1000, C=1.0)
        clf.fit(X_train, y_train[:len(X_train)])
        lda_dir = clf.coef_[0].copy()
        lda_dir /= (np.linalg.norm(lda_dir) + 1e-8)
        lda_acc = float(clf.score(X_test, y_test[:len(X_test)]))
    except:
        lda_dir = direction.copy()
        lda_acc = mean_acc

    mean_lda_cos = float(np.dot(direction, lda_dir))

    # 5. Per-sample projections
    proj_c = ac @ direction
    proj_w = aw @ direction
    gap = float(proj_w.mean() - proj_c.mean())

    # 6. Random baseline
    rand_accs = []
    for _ in range(n_random):
        rd = rng.randn(d).astype(np.float32)
        rd /= (np.linalg.norm(rd) + 1e-8)
        rc, rw = ac[idx[h:2*h]] @ rd, aw[idx[h:2*h]] @ rd
        rt = (rc.mean() + rw.mean()) / 2
        rand_accs.append(float(((rc < rt).mean() + (rw >= rt).mean()) / 2))

    return {
        "layer": layer, "direction": direction, "lda_direction": lda_dir,
        "stability": stability, "mean_acc": mean_acc, "lda_acc": lda_acc,
        "mean_lda_cos": mean_lda_cos, "gap": gap,
        "random_acc": float(np.mean(rand_accs)),
        "signal": mean_acc - float(np.mean(rand_accs)),
        "proj_c": proj_c, "proj_w": proj_w,
    }

# Run analysis
print(f"{'Layer':>6s}  {'Stab':>6s}  {'MeanAcc':>8s}  {'LDAAcc':>8s}  {'RandAcc':>8s}  {'M↔L':>6s}  {'Signal':>8s}")
print("-" * 68)

results = []
for l in range(N_LAYERS):
    r = analyze_layer(l, activations[l][correct_mask], activations[l][wrong_mask])
    results.append(r)
    marker = ""
    if r["stability"] > 0.7 and r["mean_acc"] > 0.8 and r["mean_lda_cos"] > 0.7:
        marker = " *** STRONG (mean≈LDA)"
    elif r["stability"] > 0.7 and r["mean_acc"] > 0.8:
        marker = " ** STRONG"
    elif r["signal"] > 0.1:
        marker = " * signal"
    print(f"  {l:4d}  {r['stability']:>6.3f}  {r['mean_acc']:>8.3f}  {r['lda_acc']:>8.3f}  "
          f"{r['random_acc']:>8.3f}  {r['mean_lda_cos']:>6.3f}  {r['signal']:>8.3f}{marker}")

best = max(results, key=lambda r: r["stability"] * r["mean_acc"])
print(f"\nBest layer: {best['layer']}")
print(f"  Stability:    {best['stability']:.4f}")
print(f"  Mean-diff:    {best['mean_acc']:.4f}")
print(f"  LDA:          {best['lda_acc']:.4f}")
print(f"  Random:       {best['random_acc']:.4f}")
print(f"  Signal:       {best['signal']:.4f}")
print(f"  Mean↔LDA cos: {best['mean_lda_cos']:.4f}")

## 4. Diagnostic Plots

In [ ]:
import matplotlib.pyplot as plt

layers = [r["layer"] for r in results]
stabs = [r["stability"] for r in results]
mean_accs = [r["mean_acc"] for r in results]
lda_accs = [r["lda_acc"] for r in results]
rand_accs = [r["random_acc"] for r in results]
signals = [r["signal"] for r in results]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f"Experiment 0: Does Φ Exist? — {MODEL_NAME}", fontsize=14, fontweight="bold")

# 1. Stability
ax = axes[0,0]
colors = ["#e74c3c" if s > 0.7 else "#3498db" if s > 0.5 else "#95a5a6" for s in stabs]
ax.bar(layers, stabs, color=colors)
ax.axhline(y=0.7, color="red", linestyle="--", alpha=0.7, label="threshold")
ax.set_xlabel("Layer"); ax.set_ylabel("Split-Half Cosine")
ax.set_title("Direction Stability"); ax.legend()

# 2. Real vs Random (3 methods)
ax = axes[0,1]
x = np.arange(len(layers))
w = 0.25
ax.bar(x-w, mean_accs, w, label="Mean-diff", color="#2ecc71")
ax.bar(x, lda_accs, w, label="LDA", color="#3498db")
ax.bar(x+w, rand_accs, w, label="Random", color="#e74c3c", alpha=0.7)
ax.axhline(y=0.5, color="gray", linestyle=":", alpha=0.5)
ax.set_xlabel("Layer"); ax.set_ylabel("Probe Accuracy")
ax.set_title("Mean-diff vs LDA vs Random"); ax.legend(fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(layers)

# 3. Signal (effect size)
ax = axes[1,0]
colors = ["#27ae60" if s > 0.2 else "#f39c12" if s > 0.1 else "#95a5a6" for s in signals]
ax.bar(layers, signals, color=colors)
ax.axhline(y=0.2, color="green", linestyle="--", alpha=0.5, label="strong")
ax.axhline(y=0, color="black", linewidth=0.5)
ax.set_xlabel("Layer"); ax.set_ylabel("Signal (acc - random)")
ax.set_title("How Much Better Than Random?"); ax.legend()

# 4. Mean↔LDA agreement
ax = axes[1,1]
ml_cos = [r["mean_lda_cos"] for r in results]
colors = ["#27ae60" if c > 0.7 else "#f39c12" if c > 0.3 else "#e74c3c" for c in ml_cos]
ax.bar(layers, ml_cos, color=colors)
ax.axhline(y=0.7, color="green", linestyle="--", alpha=0.5, label="agree")
ax.set_xlabel("Layer"); ax.set_ylabel("Cosine(mean-diff, LDA)")
ax.set_title("Do Two Methods Find Same Direction?"); ax.legend()

plt.tight_layout()
plt.savefig("e00_01_layer_sensitivity.png", dpi=150)
plt.show()

In [ ]:
# Projection distribution at best layer
fig, ax = plt.subplots(1, 1, figsize=(10, 6))

ax.hist(best["proj_c"], bins=50, alpha=0.6, color="#2ecc71",
        label=f"Correct (n={len(best['proj_c'])})", density=True)
ax.hist(best["proj_w"], bins=50, alpha=0.6, color="#e74c3c",
        label=f"Wrong (n={len(best['proj_w'])})", density=True)
ax.axvline(x=best["proj_c"].mean(), color="#27ae60", linestyle="--", lw=2)
ax.axvline(x=best["proj_w"].mean(), color="#c0392b", linestyle="--", lw=2)

ax.set_xlabel("Projection onto Φ direction")
ax.set_ylabel("Density")
ax.set_title(f"Projection Distribution — Layer {best['layer']}\n"
             f"Gap={best['gap']:.4f}, Acc={best['mean_acc']:.3f}, Signal={best['signal']:.3f}")
ax.legend()
plt.tight_layout()
plt.savefig("e00_02_projection_distribution.png", dpi=150)
plt.show()

## 5. Cross-Operation Generalization

In [ ]:
best_act = activations[best["layer"]]
ops = ["addition", "multiplication", "subtraction"]

# Per-operation directions
op_dirs = {}
for op in ops:
    mc = correct_mask & op_masks[op]
    mw = wrong_mask & op_masks[op]
    d = (best_act[mw].mean(0) - best_act[mc].mean(0)).numpy()
    op_dirs[op] = d / (np.linalg.norm(d) + 1e-8)

# Transfer matrix
gen_matrix = np.eye(3)
gen_results = {}
for i, train_op in enumerate(ops):
    for j, test_op in enumerate(ops):
        if i == j: continue
        tc = correct_mask & op_masks[test_op]
        tw = wrong_mask & op_masks[test_op]
        pc = best_act[tc].numpy() @ op_dirs[train_op]
        pw = best_act[tw].numpy() @ op_dirs[train_op]
        t = (pc.mean() + pw.mean()) / 2
        acc = float(((pc < t).mean() + (pw >= t).mean()) / 2)
        gen_matrix[i, j] = acc
        gen_results[f"{train_op}→{test_op}"] = acc

# Cross-direction cosines
print("Cross-direction cosines:")
cross_cos = {}
for i, o1 in enumerate(ops):
    for o2 in ops[i+1:]:
        c = float(np.dot(op_dirs[o1], op_dirs[o2]))
        cross_cos[f"{o1}↔{o2}"] = c
        print(f"  cos({o1[:4]}, {o2[:4]}) = {c:.3f}")

mean_cross = np.mean(list(cross_cos.values()))
print(f"  Mean: {mean_cross:.3f}")
if mean_cross > 0.7:
    print("  → Same underlying phenomenon")
elif mean_cross > 0.3 and all(v > 0.65 for v in gen_results.values()):
    print("  → DEEPER STRUCTURE: different dirs but transfer works!")

# Plot
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
im = ax.imshow(gen_matrix, cmap="RdYlGn", vmin=0.4, vmax=1.0)
ax.set_xticks(range(3)); ax.set_yticks(range(3))
ax.set_xticklabels([o[:4] for o in ops])
ax.set_yticklabels([o[:4] for o in ops])
ax.set_xlabel("Test"); ax.set_ylabel("Train")
ax.set_title(f"Cross-Operation Generalization — Layer {best['layer']}")
for i in range(3):
    for j in range(3):
        color = "white" if gen_matrix[i,j] < 0.65 else "black"
        ax.text(j, i, f"{gen_matrix[i,j]:.2f}", ha="center", va="center",
                color=color, fontweight="bold", fontsize=14)
plt.colorbar(im, ax=ax, label="Accuracy")
plt.tight_layout()
plt.savefig("e00_03_generalization.png", dpi=150)
plt.show()

## 6. Cross-Domain Transfer: Arithmetic Φ → Factual Claims

In [ ]:
# Does the arithmetic error direction also separate real from fake facts?
cite_samples = generate_citation_data(n=50)
cite_prompts = [s.prompt for s in cite_samples]
cite_correct = [s.is_correct for s in cite_samples]

cite_acts = {best["layer"]: []}
for batch_start in range(0, len(cite_prompts), BATCH_SIZE):
    batch = cite_prompts[batch_start:batch_start+BATCH_SIZE]
    _, cache = model.run_with_cache(
        batch,
        names_filter=lambda name: f"blocks.{best['layer']}.hook_resid_post" in name,
    )
    cite_acts[best["layer"]].append(
        cache[f"blocks.{best['layer']}.hook_resid_post"][:, -1, :].float().cpu()
    )
    del cache

cite_act = torch.cat(cite_acts[best["layer"]], dim=0)
cite_mask = torch.tensor(cite_correct)

# Apply arithmetic Φ to citations
cite_c = cite_act[cite_mask].numpy() @ best["direction"]
cite_w = cite_act[~cite_mask].numpy() @ best["direction"]
t = (cite_c.mean() + cite_w.mean()) / 2
cite_acc = float(((cite_c < t).mean() + (cite_w >= t).mean()) / 2)

print(f"\nArithmetic Φ → Citation domain: accuracy = {cite_acc:.3f}")
if cite_acc > 0.65:
    print("CROSS-DOMAIN TRANSFER WORKS")
    print("→ Φ is NOT just arithmetic error — it's a general error direction!")
elif cite_acc > 0.55:
    print("Weak transfer — some shared structure")
else:
    print("No transfer — Φ is arithmetic-specific")

# Plot
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
ax.hist(cite_c, bins=25, alpha=0.6, color="#2ecc71", label="Real facts", density=True)
ax.hist(cite_w, bins=25, alpha=0.6, color="#e74c3c", label="Fake facts", density=True)
ax.set_xlabel("Projection onto arithmetic Φ")
ax.set_ylabel("Density")
ax.set_title(f"Cross-Domain Transfer: Arithmetic Φ on Factual Claims\n"
             f"Separation accuracy = {cite_acc:.3f}")
ax.legend()
plt.tight_layout()
plt.savefig("e00_04_cross_domain.png", dpi=150)
plt.show()

## 7. All-Layer Distribution Overview

In [ ]:
n_cols = min(6, N_LAYERS)
n_rows = (N_LAYERS + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(3*n_cols, 2.5*n_rows))
if n_rows == 1: axes = [axes]
fig.suptitle("Projection Distributions per Layer", fontsize=12, fontweight="bold")

for idx, r in enumerate(results):
    row, col = idx // n_cols, idx % n_cols
    ax = axes[row][col] if isinstance(axes[row], (list, np.ndarray)) else axes[row]
    ax.hist(r["proj_c"], bins=30, alpha=0.5, color="#2ecc71", density=True)
    ax.hist(r["proj_w"], bins=30, alpha=0.5, color="#e74c3c", density=True)
    m = "**" if r["stability"] > 0.7 and r["mean_acc"] > 0.8 else ""
    ax.set_title(f"L{r['layer']} {m}\nacc={r['mean_acc']:.2f}", fontsize=8)
    ax.tick_params(labelsize=6)

for idx in range(len(results), n_rows * n_cols):
    row, col = idx // n_cols, idx % n_cols
    ax = axes[row][col] if isinstance(axes[row], (list, np.ndarray)) else axes[row]
    ax.set_visible(False)

plt.tight_layout()
plt.savefig("e00_05_all_layers.png", dpi=150)
plt.show()

## 8. VERDICT

In [ ]:
signal = best["signal"]
methods_agree = best["mean_lda_cos"] > 0.7
phi_exists = best["stability"] > 0.7 and best["mean_acc"] > 0.8
phi_better = signal > 0.2
phi_generalizes = all(v > 0.65 for v in gen_results.values())
phi_cross_domain = cite_acc > 0.6

print("=" * 60)
print("EXPERIMENT 0: VERDICT")
print("=" * 60)
print(f"\n  {'✓' if phi_exists else '✗'} Direction stable & separable (stab={best['stability']:.3f}, acc={best['mean_acc']:.3f})")
print(f"  {'✓' if phi_better else '✗'} Signal > 0.2 above random (signal={signal:.3f})")
print(f"  {'✓' if methods_agree else '✗'} Mean-diff ≈ LDA (cos={best['mean_lda_cos']:.3f})")
print(f"  {'✓' if phi_generalizes else '✗'} Cross-operation generalization")
print(f"  {'✓' if phi_cross_domain else '✗'} Cross-domain transfer to citations ({cite_acc:.3f})")

n_passed = sum([phi_exists, phi_better, methods_agree, phi_generalizes, phi_cross_domain])
print(f"\n  Score: {n_passed}/5")

if n_passed == 5:
    print("\n  ★★★ VERY STRONG: Φ EXISTS, GENERALIZES, TRANSFERS ACROSS DOMAINS")
    print("  → Proceed to Experiment 1 (projection + sign flip test)")
elif n_passed >= 3:
    print(f"\n  ★★ STRONG: Φ exists with {5-n_passed} caveat(s)")
    print("  → Proceed to Experiment 1, investigate weak points")
elif n_passed >= 2:
    print(f"\n  ★ WEAK SIGNAL: something is there but not clean")
    print("  → Try larger model, more data, or SAE decomposition")
else:
    print(f"\n  ✗ NOT FOUND")
    print("  → Try multi-direction Φ, SAE features, or nonlinear probe")

## 9. Save Results for Experiment 1

In [ ]:
import json

# Save best direction
np.save(f"phi_direction_layer{best['layer']}_{MODEL_NAME}.npy", best["direction"])
np.save(f"phi_lda_direction_layer{best['layer']}_{MODEL_NAME}.npy", best["lda_direction"])

# Save JSON summary
summary = {
    "model": MODEL_NAME, "best_layer": best["layer"],
    "stability": best["stability"], "mean_acc": best["mean_acc"],
    "lda_acc": best["lda_acc"], "random_acc": best["random_acc"],
    "signal": best["signal"], "mean_lda_cos": best["mean_lda_cos"],
    "gap": best["gap"], "generalization": gen_results,
    "cross_cosines": cross_cos, "citation_transfer": cite_acc,
    "score": f"{n_passed}/5",
    "per_layer": [{"layer": r["layer"], "stability": r["stability"],
                    "mean_acc": r["mean_acc"], "lda_acc": r["lda_acc"],
                    "random_acc": r["random_acc"], "signal": r["signal"],
                    "mean_lda_cos": r["mean_lda_cos"]} for r in results],
}
with open(f"e00_results_{MODEL_NAME}.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"Saved:")
print(f"  phi_direction_layer{best['layer']}_{MODEL_NAME}.npy")
print(f"  phi_lda_direction_layer{best['layer']}_{MODEL_NAME}.npy")
print(f"  e00_results_{MODEL_NAME}.json")
print(f"\nDownload these files for Experiment 1.")